# Sensitive Court Data AI Triage & Governance Evaluation

This notebook treats all source records as if they were highly sensitive real court data. The dataset included in the repository is synthetic, but every processing step follows a privacy/security/public-safety design.

## 1. Governance Principles

- Privacy by design: classify and redact before modeling.
- Security by design: pseudonymized case keys and raw text exclusion from public-safe outputs.
- Public safety: threat, weapon, emergency, protection-order, minor/guardian, and sealed-record signals trigger mandatory review.
- Human authority: AI recommends routing; humans make final operational decisions.
- Scope restriction: no legal outcome, guilt, sentencing, eligibility, or legal-advice prediction.

In [1]:
import pandas as pd, numpy as np, re, hashlib, json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('/mnt/data/am_sensitive_court_ai_governance')
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
DOCS = ROOT / 'docs'
pd.set_option('display.max_colwidth', 140)


In [2]:
processed = pd.read_csv(DATA / 'processed_redacted_governed_case_dataset.csv')
metrics = pd.read_csv(OUT / 'model_metrics_governed.csv')
impact = pd.read_csv(OUT / 'governed_impact_summary.csv')
audit = pd.read_csv(OUT / 'audit_log_sample.csv')
queue = pd.read_csv(OUT / 'human_review_queue_sample.csv')
report = json.loads((OUT / 'data_quality_security_report.json').read_text())
report

{'raw_rows': 8096,
 'missing_source_channel': 123,
 'duplicate_case_text_rows': 96,
 'security_assumption': 'Treat all source data as highly sensitive real court data; raw text is quarantined and never used directly for modeling output artifacts.',
 'clean_rows': 8000,
 'rows_removed_as_duplicates': 96,
 'pii_tokens_detected_total': 14442,
 'restricted_records': 6401}

## 2. Data Quality + Security Report

The pipeline reports missing channels, duplicate records, PII-token detection, restricted-record volume, and row counts after duplicate removal.

In [3]:
pd.DataFrame([report]).T.rename(columns={0:'value'})

,value
raw_rows,8096
missing_source_channel,123
duplicate_case_text_rows,96
security_assumption,Treat all source data as highly sensitive real court data; raw text is quarantined and never used directly for modeling output artifacts.
clean_rows,8000
rows_removed_as_duplicates,96
pii_tokens_detected_total,14442
restricted_records,6401


## 3. Public-Safe Dataset

The public-safe dataset excludes direct case IDs and raw sensitive text. Modeling is performed on redacted normalized text.

In [4]:
safe_cols = ['case_key_hash','filing_date','county','case_title','case_description_redacted','case_category','case_subcategory','sensitivity_classification','pii_total','public_safety_signal']
processed[safe_cols].head(8)

,case_key_hash,filing_date,county,case_title,case_description_redacted,case_category,case_subcategory,sensitivity_classification,pii_total,public_safety_signal
0,30d9d2bdb37cf2902c817ca0,2024-09-27,Marion,Property Dispute intake review - Marion County,"Allen County intake matter received for operational review. The case is at expedited clerk review; key signals include parcel map, easem...",Civil,Property Dispute,Restricted-Court Data,4,0
1,ebe438bf53c0a8113eba8dd5,2024-05-10,Tippecanoe,Protection Order intake review - Tippecanoe County,"Lake County intake matter received for operational review. The case is at hearing prep; key signals include emergency hearing, credible ...",Domestic/Family,Protection Order,Restricted-Court Data,2,1
2,fcb68555097292cd93844522,2024-10-03,St. Joseph,Felony intake review - St. Joseph County,"Vanderburgh County intake matter received for operational review. The case is at motion review; key signals include arraignment, chargin...",Criminal,Felony,Restricted-Court Data,3,1
3,7df287b3205d03cabdb593f2,2024-06-15,Johnson,Guardianship intake review - Johnson County,"Monroe County intake matter received for operational review. The case is at post-filing review; key signals include minor child, guardia...",Domestic/Family,Guardianship,Restricted-Court Data,0,0
4,3a3c027f72bba651dde5d3f6,2024-09-27,Elkhart,Traffic Appeal intake review - Elkhart County,"Monroe County intake matter received for operational review. The case is at hearing prep; key signals include license points, appeal dea...",Administrative,Traffic Appeal,Restricted-Court Data,0,0
5,8bd41c3b00c25d04d52281f1,2024-12-03,Hamilton,Personal Injury intake review - Hamilton County,"Porter County intake matter received for operational review. The case is at post-filing review; key signals include medical bills, injur...",Civil,Personal Injury,Restricted-Court Data,3,0
6,7e85775b5880704691bc12eb,2024-02-17,Porter,Property Dispute intake review - Porter County,"Tippecanoe County intake matter received for operational review. The case is at initial intake; key signals include injunction request, ...",Civil,Property Dispute,Confidential,2,1
7,0c2c4036548e9ac8e3ca4976,2024-08-28,Porter,Compliance Review intake review - Porter County,"Tippecanoe County intake matter received for operational review. The case is at case review; key signals include remediation plan, contr...",Administrative,Compliance Review,Confidential,2,1


## 4. Sensitivity and PII Controls

In [5]:
processed['sensitivity_classification'].value_counts().to_frame('records')

,records
sensitivity_classification,
Restricted-Court Data,6401
Confidential,875
Internal,724


In [6]:
processed[['pii_ssn','pii_phone','pii_email','pii_dob','pii_street_address','pii_person_name_context','pii_total']].sum().to_frame('detected_tokens')

,detected_tokens
pii_ssn,1653
pii_phone,3180
pii_email,1653
pii_dob,1589
pii_street_address,3180
pii_person_name_context,3187
pii_total,14442


## 5. Model Benchmarking

TF-IDF is positioned as the controlled pilot model because it is transparent, fast, and easier to govern. The BERT-style semantic benchmark is shadow mode only.

In [7]:
metrics

,model,use_case,accuracy,macro_f1,mean_confidence,status
0,TF-IDF + Logistic Regression,Baseline category routing,1.0,1.0,0.957792,Recommended controlled pilot
1,TF-IDF + LinearSVC,Subcategory routing,1.0,1.0,NaN,Specialized routing model
2,BERT-style semantic benchmark (offline shadow),Advanced semantic benchmark,1.0,1.0,0.961446,Shadow mode only


## 6. Guardrail Decisions

Records are not simply routed based on the model prediction. Restricted data, public-safety signals, low confidence, and poor label quality force mandatory human review.

In [8]:
processed_scored = pd.read_csv(OUT / 'scored_cases_with_guardrails.csv')
processed_scored['guardrail_action'].value_counts().to_frame('records')

,records
guardrail_action,
Mandatory Human Review,7278
AI-Assisted Routing Candidate,722


In [9]:
processed_scored[['case_key_hash','case_category','predicted_category','prediction_confidence','sensitivity_classification','public_safety_signal','guardrail_action','guardrail_reason']].head(12)

,case_key_hash,case_category,predicted_category,prediction_confidence,sensitivity_classification,public_safety_signal,guardrail_action,guardrail_reason
0,30d9d2bdb37cf2902c817ca0,Civil,Civil,0.960,Restricted-Court Data,0,Mandatory Human Review,restricted data
1,ebe438bf53c0a8113eba8dd5,Domestic/Family,Domestic/Family,0.980,Restricted-Court Data,1,Mandatory Human Review,restricted data; public safety signal
2,fcb68555097292cd93844522,Criminal,Criminal,0.980,Restricted-Court Data,1,Mandatory Human Review,restricted data; public safety signal
3,7df287b3205d03cabdb593f2,Domestic/Family,Domestic/Family,0.956,Restricted-Court Data,0,Mandatory Human Review,restricted data
4,3a3c027f72bba651dde5d3f6,Administrative,Administrative,0.968,Restricted-Court Data,0,Mandatory Human Review,restricted data
5,8bd41c3b00c25d04d52281f1,Civil,Civil,0.960,Restricted-Court Data,0,Mandatory Human Review,restricted data
6,7e85775b5880704691bc12eb,Civil,Civil,0.950,Confidential,1,Mandatory Human Review,public safety signal
7,0c2c4036548e9ac8e3ca4976,Administrative,Administrative,0.954,Confidential,1,Mandatory Human Review,public safety signal
8,e05c2f3dbff6d51d67f4dbbb,Criminal,Criminal,0.905,Restricted-Court Data,0,Mandatory Human Review,restricted data
9,c3d25c02d38ac3c0aa507d9c,Criminal,Criminal,0.956,Restricted-Court Data,0,Mandatory Human Review,restricted data


## 7. Human Review Queue

This is the operational queue a reviewer would see. The queue is ordered around restricted records, public-safety indicators, and low-confidence cases.

In [10]:
queue[['case_key_hash','case_category','predicted_category','prediction_confidence','sensitivity_classification','public_safety_signal','guardrail_reason']].head(15)

,case_key_hash,case_category,predicted_category,prediction_confidence,sensitivity_classification,public_safety_signal,guardrail_reason
0,30d9d2bdb37cf2902c817ca0,Civil,Civil,0.960,Restricted-Court Data,0,restricted data
1,ebe438bf53c0a8113eba8dd5,Domestic/Family,Domestic/Family,0.980,Restricted-Court Data,1,restricted data; public safety signal
2,fcb68555097292cd93844522,Criminal,Criminal,0.980,Restricted-Court Data,1,restricted data; public safety signal
3,7df287b3205d03cabdb593f2,Domestic/Family,Domestic/Family,0.956,Restricted-Court Data,0,restricted data
4,3a3c027f72bba651dde5d3f6,Administrative,Administrative,0.968,Restricted-Court Data,0,restricted data
5,8bd41c3b00c25d04d52281f1,Civil,Civil,0.960,Restricted-Court Data,0,restricted data
6,7e85775b5880704691bc12eb,Civil,Civil,0.950,Confidential,1,public safety signal
7,0c2c4036548e9ac8e3ca4976,Administrative,Administrative,0.954,Confidential,1,public safety signal
8,e05c2f3dbff6d51d67f4dbbb,Criminal,Criminal,0.905,Restricted-Court Data,0,restricted data
9,c3d25c02d38ac3c0aa507d9c,Criminal,Criminal,0.956,Restricted-Court Data,0,restricted data


## 8. Auditability

The audit log captures enough evidence to reconstruct the AI-assisted decision path without storing raw narratives in downstream dashboards.

In [11]:
audit.head(10)

,event_timestamp,case_key_hash,raw_text_sha256,model_version,predicted_category,prediction_confidence,sensitivity_classification,public_safety_signal,guardrail_action,guardrail_reason,actor_role,reviewer_decision,audit_event_type
0,2025-01-15 09:00:00,72a420a405cd979beb6e40ec,c892c1cd288cded53553b05595a87f4bf76410100f24d4f21f46450156395df0,tfidf-lg-v1.0-redacted-inputs,Civil,0.963,Internal,0,AI-Assisted Routing Candidate,confidence above threshold and no restricted/public-safety flags,AI Triage Service,pending reviewer sampling,ai_assisted_route_suggested
1,2025-01-15 09:17:00,c4252409fbe448d5b600c44c,73a1bcf44ad74a39274f038d998a6272a8630113df8790d61fa1db937262c7dd,tfidf-lg-v1.0-redacted-inputs,Civil,0.934,Restricted-Court Data,0,Mandatory Human Review,restricted data,Court Operations Reviewer,overrode category,human_review_required
2,2025-01-15 09:34:00,3e17b9b8dd585b9a7fb52461,da508fc756d14ae1fd07d179d63e449ce6f2a981be74d7c751de862a86cef15a,tfidf-lg-v1.0-redacted-inputs,Civil,0.945,Confidential,1,Mandatory Human Review,public safety signal,Court Operations Reviewer,held for sealed-record review,human_review_required
3,2025-01-15 09:51:00,f02fd2c06beb1d941f474bd8,82252e172716069d41bbcbfec0a6e99f6c7fdd948d0f893d86a48aec049ba01e,tfidf-lg-v1.0-redacted-inputs,Civil,0.960,Confidential,1,Mandatory Human Review,public safety signal,Court Operations Reviewer,confirmed prediction,human_review_required
4,2025-01-15 10:08:00,5028bb26d535deb93d454b5c,272953eab8b3111d93a28b9b6acae731efaee9c16b1e3e0d7d4b7232138fe9ad,tfidf-lg-v1.0-redacted-inputs,Domestic/Family,0.957,Restricted-Court Data,0,Mandatory Human Review,restricted data,Court Operations Reviewer,requested legal staff review,human_review_required
5,2025-01-15 10:25:00,1655855c990b9f032e188c2b,5868b2fd5dd2b64212068436cfc5c0c19e21089571f28f4c0acb59f9b29df7cd,tfidf-lg-v1.0-redacted-inputs,Criminal,0.968,Restricted-Court Data,1,Mandatory Human Review,restricted data; public safety signal,Court Operations Reviewer,confirmed prediction,human_review_required
6,2025-01-15 10:42:00,5f9ccba1f1b187b192cf2380,fb7678dde83458ab667aba46759977325ff233715891675da3bd1e4d98c27acd,tfidf-lg-v1.0-redacted-inputs,Domestic/Family,0.961,Restricted-Court Data,1,Mandatory Human Review,restricted data; public safety signal,Court Operations Reviewer,confirmed prediction,human_review_required
7,2025-01-15 10:59:00,968575868b33fe9894997b4e,7df7f2cebfc1b9c9b608019197cc4b797d5ccb5b2c7928d1bdb0cedf8b2db7dc,tfidf-lg-v1.0-redacted-inputs,Criminal,0.926,Restricted-Court Data,1,Mandatory Human Review,restricted data; public safety signal; label needs review,Court Operations Reviewer,overrode category,human_review_required
8,2025-01-15 11:16:00,d68fc7b6ab924f07d50f26ce,89e8995ec80566cdef78a146c5c022dadcad75b417db47c1fa4e90f096a69ea1,tfidf-lg-v1.0-redacted-inputs,Civil,0.965,Confidential,1,Mandatory Human Review,public safety signal,Court Operations Reviewer,held for sealed-record review,human_review_required
9,2025-01-15 11:33:00,a820a7b3880f24bed4e9e688,72cc298d242f26267d40672b18547d936498513f1128a1144a7101f5348c6c3f,tfidf-lg-v1.0-redacted-inputs,Administrative,0.975,Restricted-Court Data,0,Mandatory Human Review,restricted data,Court Operations Reviewer,requested legal staff review,human_review_required


## 9. Governed Impact

Efficiency is calculated only after respecting mandatory human-review controls. Savings are not claimed on all records.

In [12]:
impact

,case_category,cases,manual_hours,mandatory_review,restricted,public_safety,ai_assist_rate,projected_hours_saved
0,Administrative,920,234.3,776,633,323,0.156522,11.7
1,Civil,2391,766.9,2036,1596,800,0.148473,36.4
2,Criminal,2422,989.9,2267,2073,1017,0.063997,20.3
3,Domestic/Family,2267,1023.6,2199,2099,1122,0.029996,9.8


## 10. Executive Recommendation

Proceed with a limited pilot of the TF-IDF routing model using redacted inputs, mandatory human review, and audit logging. Keep the BERT-style semantic benchmark in shadow mode. Do not scale until privacy, security, public-safety, model-performance, and reviewer-adoption gates pass.